# 01 · Game basics

Welcome. This notebook is for someone who has never touched the framework before. By the end you will know how to create a Hex Connect-6 game, place stones, detect a win, undo moves, and read the board state programmatically.

Estimated time: 10 minutes.

Prerequisites: `pip install hexbot` (the first cell below does this if you are running in Colab).

In [ ]:
# Install (skip this cell if hexbot is already installed)
!pip install --quiet hexbot

## What is Hex Connect-6?

An abstract two-player game on an **infinite hexagonal grid**.

- **Goal:** be first to place six of your stones in a straight line (horizontal, diagonal, or anti-diagonal).
- **Turn structure:** Player 0 places 1 stone on turn 1, then both players alternate placing 2 stones per turn forever after. (Sometimes written 1-2-2.)
- **Board:** infinite, so there is no edge to play to. In practice the play stays within a few rings of the centre because lines have to connect.

The framework calls Player 0 "P0" and Player 1 "P1". Internally a third axis `s = -q - r` is implied but you only ever specify `(q, r)`.

See the [Game Rules wiki page](https://github.com/Saiki77/hexbot-building-framework/wiki/Game-Rules) for a deeper treatment.

## Creating a game

In [ ]:
from hexbot import HexGame

game = HexGame()
print(f"total stones: {game.total_stones}")
print(f"current player: {game.current_player}")
print(f"stones to place this turn: {game.stones_per_turn}")

## Placing stones

`game.place(q, r)` puts a stone for the current player. The framework handles the 1-2-2 turn structure automatically: it knows when a turn ends and switches the current player for you.

In [ ]:
game = HexGame()

# Turn 1: P0 places one stone
game.place(0, 0)
print(f"after P0 plays once:    current={game.current_player}")

# Turn 2: P1 places two stones
game.place(2, 0)
print(f"after P1's first stone: current={game.current_player}  (still P1)")
game.place(2, -1)
print(f"after P1's second:      current={game.current_player}  (back to P0)")

print(f"\ntotal stones placed: {game.total_stones}")

## Win detection

`game.winner` returns `0`, `1`, or `None`. `game.is_over` is the boolean version. The framework checks all three line directions automatically after each placement.

In [ ]:
# A scripted P0 win along the q axis
g = HexGame()
g.place(0, 0)                 # P0
g.place(0, 5); g.place(0, 6)  # P1, far away
g.place(1, 0); g.place(2, 0)  # P0
g.place(1, 5); g.place(1, 6)  # P1
g.place(3, 0); g.place(4, 0)  # P0
g.place(2, 5); g.place(2, 6)  # P1
g.place(5, 0)                 # P0 wins: (0,0)..(5,0)

print(f"is_over: {g.is_over}")
print(f"winner:  P{g.winner}")

## Undoing moves

Every stone you place can be undone. This is what lets search algorithms (alpha-beta, MCTS) explore without copying the whole board.

In [ ]:
g = HexGame()
g.place(0, 0)
g.place(1, 0)
g.place(1, -1)
print(f"after 3 stones: total={g.total_stones}")

g.undo()
print(f"after undo:     total={g.total_stones}  (last stone removed)")

g.undo(); g.undo()
print(f"after 2 more:   total={g.total_stones}  (back to empty board)")

## Reading the board

These are the most common read-only queries you will use.

In [ ]:
g = HexGame()
for q, r in [(0,0), (3,0), (3,-1), (1,0), (2,0)]:
    g.place(q, r)

# Where can the next stone legally go?
moves = g.legal_moves()
print(f"legal moves: {len(moves)} cells")

# Top 5 candidates ranked by the C engine's heuristic
top = g.scored_moves(5)
print("\ntop 5 by C heuristic (higher = better):")
for q, r, score in top:
    print(f"  ({q:>3},{r:>3})  score={score}")

## Cloning a game

Sometimes you need an independent copy (for example to try a hypothetical line in your own search code).

In [ ]:
g = HexGame()
g.place(0, 0)

twin = g.clone()
twin.place(1, 0); twin.place(1, -1)

print(f"original: {g.total_stones} stones")
print(f"twin:     {twin.total_stones} stones")
print("clones are independent.")

## Try it yourself

Two short exercises to make this stick:

In [ ]:
# 1. Build a game where P1 wins on the diagonal direction (1, -1).
#    Hint: use stones at (k, -k) for k = 0..5 with P1 to move at the end.
#    Remember the 1-2-2 turn structure when sequencing P0's responses.

g = HexGame()
# TODO: fill in the sequence of g.place(q, r) calls.
# When done, this should print: "P1 wins"
if g.is_over:
    print(f"P{g.winner} wins")
else:
    print("game still in progress")

In [ ]:
# 2. Without calling .clone(), use undo() to evaluate which of two candidate
#    moves looks better according to scored_moves's top score after each.

g = HexGame()
g.place(0, 0)
g.place(1, 0)  # P1 first stone

candidates = [(2, -1), (-1, 1)]
for q, r in candidates:
    g.place(q, r)
    top_score = g.scored_moves(1)[0][2]
    print(f"after P1 plays ({q},{r}): best follow-up score = {top_score}")
    g.undo()

## Next

You can now drive the game engine. The next notebook covers the analysis tools (threats, alpha-beta search, the endgame solver) and shows how to write your first bot.

→ [02 · Analysis tools and writing a bot](02_analysis_and_bots.ipynb)

Or jump to the [Bot Approaches wiki page](https://github.com/Saiki77/hexbot-building-framework/wiki/Bot-Approaches) for six fully worked bot designs.